# 🫀 Hybrid Explainable Cardiovascular Intelligence System
### University-Grade Medical AI Pipeline

**Architecture Overview:**
```
Cell 1  → Setup, Drive Mount, Package Install
Cell 2  → Dataset Download (KaggleHub)
Cell 3  → Exploratory Data Analysis (EDA)
Cell 4  → Data Cleaning & Repair   → saves cleaned CSVs
Cell 5  → Medical Feature Engineering → saves engineered CSVs
Cell 6  → Model Training (XGB, CatBoost, LGBM, LR)
Cell 7  → Ensemble & Stacking
Cell 8  → SHAP Explainability
Cell 9  → Robust Inference Pipeline (missing-value tolerant)
Cell 10 → Medical Report Generator
```

> **Design note:** The ML pipeline exposes a clean `predict_patient()` interface  
> that is ready to **fuse with the Fuzzy Logic engine** in a future step.
> Fuzzy inputs (echocardiography measurements) arrive through a separate channel.


In [ ]:
# ============================================================
# CELL 1 — Setup, Drive Mount, Package Installation
# ============================================================

# ── 1.1  Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/CardioAI'
PATHS = {
    'raw'       : f'{BASE_DIR}/data/raw',
    'cleaned'   : f'{BASE_DIR}/data/cleaned',
    'engineered': f'{BASE_DIR}/data/engineered',
    'models'    : f'{BASE_DIR}/models',
    'reports'   : f'{BASE_DIR}/reports',
    'plots'     : f'{BASE_DIR}/plots',
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
print('✅ Drive mounted. Folder structure ready.')
for k, v in PATHS.items():
    print(f'   {k:12s} → {v}')

# ── 1.2  Package Installation ──────────────────────────────
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install('kagglehub')
pip_install('xgboost', 'catboost', 'lightgbm')
pip_install('shap')
pip_install('scikit-learn', 'imbalanced-learn')
pip_install('matplotlib', 'seaborn', 'plotly')
print('✅ All packages installed.')

# ── 1.3  Global Imports ────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import joblib
import json
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# ── 1.4  Reproducibility ───────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── 1.5  Matplotlib style ──────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f8f9fa',
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

print('\n✅ Setup complete. Ready to proceed.')

In [ ]:
# ============================================================
# CELL 2 — Dataset Download via KaggleHub
# ============================================================

import kagglehub, shutil, glob

def download_and_copy(dataset_slug: str, dest_dir: str, label: str) -> list:
    """
    Download a Kaggle dataset and copy all CSV files to dest_dir.
    Returns list of destination file paths.
    """
    print(f'⬇️  Downloading [{label}] ...')
    path = kagglehub.dataset_download(dataset_slug)
    print(f'   Raw path: {path}')

    csv_files = glob.glob(str(Path(path) / '**' / '*.csv'), recursive=True)
    if not csv_files:
        # try xlsx as fallback
        csv_files = glob.glob(str(Path(path) / '**' / '*.xlsx'), recursive=True)

    destinations = []
    for f in csv_files:
        fname = Path(f).name
        dst   = str(Path(dest_dir) / fname)
        shutil.copy2(f, dst)
        destinations.append(dst)
        print(f'   ✓ Copied: {fname}')

    return destinations

# ── Dataset 1: Heart Disease (UCI-based) ──────────────────
hd_files = download_and_copy(
    'mahmoudshaheen1134/heart-disease-data',
    PATHS['raw'],
    'Heart Disease Dataset'
)

# ── Dataset 2: Cardiovascular ─────────────────────────────
cv_files = download_and_copy(
    'ritunandhan/cardiovascular-dataset',
    PATHS['raw'],
    'Cardiovascular Dataset'
)

# ── Auto-detect the correct files ─────────────────────────
def detect_dataset(files: list, candidate_cols: list) -> Optional[pd.DataFrame]:
    """
    Try to load each file; return the first one that contains
    at least one of the candidate columns.
    """
    for f in files:
        try:
            # Try common delimiters
            for sep in [',', ';', '\t']:
                try:
                    df = pd.read_csv(f, sep=sep, low_memory=False)
                    df.columns = df.columns.str.strip().str.lower()
                    if any(c in df.columns for c in candidate_cols):
                        print(f'   ✓ Loaded: {Path(f).name}  shape={df.shape}  sep="{sep}"')
                        return df
                except Exception:
                    continue
        except Exception as e:
            print(f'   ⚠️  Could not load {f}: {e}')
    return None

print('\n── Loading dataframes ──')
df_hd = detect_dataset(hd_files, ['target', 'trestbps', 'chol', 'thalach'])
df_cv = detect_dataset(cv_files, ['cardio', 'ap_hi', 'ap_lo', 'cholesterol'])

assert df_hd is not None, '❌ Heart Disease dataset not found. Check the Kaggle slug.'
assert df_cv is not None, '❌ Cardiovascular dataset not found. Check the Kaggle slug.'

print(f'\n✅ Heart Disease  → {df_hd.shape[0]:,} rows × {df_hd.shape[1]} cols')
print(f'✅ Cardiovascular → {df_cv.shape[0]:,} rows × {df_cv.shape[1]} cols')

In [ ]:
# ============================================================
# CELL 3 — Exploratory Data Analysis (EDA)
# ============================================================

def eda_summary(df: pd.DataFrame, name: str) -> None:
    """Print a clinical-style EDA summary for a dataframe."""
    print(f'\n{"═"*60}')
    print(f'  EDA: {name}')
    print(f'{"═"*60}')
    print(f'Shape      : {df.shape}')
    print(f'Duplicates : {df.duplicated().sum()}')
    print(f'\nNull counts:')
    nulls = df.isnull().sum()
    print(nulls[nulls > 0].to_string() if nulls.any() else '  None')
    print(f'\nDtypes:\n{df.dtypes.to_string()}')
    print(f'\nDescriptive stats:')
    display(df.describe())

def plot_eda(df: pd.DataFrame, target_col: str, name: str) -> None:
    """Plot class balance, distributions, and correlation heatmap."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'EDA — {name}', fontsize=15, fontweight='bold')

    # ── Class balance ──────────────────────────────────────
    counts = df[target_col].value_counts()
    colors = ['#2196F3', '#F44336']
    axes[0].bar(counts.index.astype(str), counts.values, color=colors)
    axes[0].set_title('Class Balance')
    axes[0].set_xlabel('Target')
    axes[0].set_ylabel('Count')
    for i, (idx, val) in enumerate(counts.items()):
        axes[0].text(i, val + counts.max() * 0.01, str(val),
                     ha='center', fontweight='bold')

    # ── Age distribution ──────────────────────────────────
    age_col = 'age' if 'age' in df.columns else df.select_dtypes('number').columns[0]
    for label, grp in df.groupby(target_col)[age_col]:
        color = '#2196F3' if label == 0 else '#F44336'
        axes[1].hist(grp, bins=25, alpha=0.6, color=color, label=f'Class {label}')
    axes[1].set_title(f'{age_col} Distribution by Class')
    axes[1].set_xlabel(age_col)
    axes[1].legend()

    # ── Correlation heatmap ───────────────────────────────
    num_cols = df.select_dtypes(include='number').columns.tolist()
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, ax=axes[2], cmap='RdBu_r',
                center=0, vmin=-1, vmax=1,
                annot=len(num_cols) <= 12,
                fmt='.1f', linewidths=0.3,
                annot_kws={'size': 7})
    axes[2].set_title('Feature Correlation')
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    save_path = f"{PATHS['plots']}/eda_{name.replace(' ', '_').lower()}.png"
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 Saved: {save_path}')

# Run EDA
eda_summary(df_hd, 'Heart Disease Dataset')
plot_eda(df_hd, 'target', 'Heart Disease')

eda_summary(df_cv, 'Cardiovascular Dataset')
plot_eda(df_cv, 'cardio', 'Cardiovascular')

In [ ]:
# ============================================================
# CELL 4 — Data Cleaning & Repair
# ============================================================
# All cleaning is documented step-by-step so every decision
# is explainable at a university presentation.
# ============================================================

class MedicalDataCleaner:
    """
    Production-grade medical data cleaning pipeline.
    Each cleaning step logs what was removed / corrected.
    """

    def __init__(self, df: pd.DataFrame, dataset_name: str):
        self.df   = df.copy()
        self.name = dataset_name
        self.log  = []
        self._original_shape = df.shape

    def _record(self, step: str, before: int, after: int) -> None:
        removed = before - after
        self.log.append({'step': step, 'before': before,
                         'after': after, 'removed': removed})
        print(f'   [{step}]  rows: {before:,} → {after:,}  (removed {removed:,})')

    # ── 4.1  Remove exact duplicates ─────────────────────
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        self._record('remove_duplicates', before, len(self.df))
        return self

    # ── 4.2  Convert age from days → years (Cardiovascular) ─
    def fix_age_days_to_years(self):
        """
        The Cardiovascular dataset stores age in DAYS.
        Realistic age in days: ~7300 (20 yr) to ~36500 (100 yr).
        Convert if median > 1000.
        """
        if 'age' not in self.df.columns:
            return self
        median_age = self.df['age'].median()
        if median_age > 1000:
            self.df['age'] = (self.df['age'] / 365.25).round(1)
            print(f'   [fix_age] Converted age from days to years. '
                  f'New median: {self.df["age"].median():.1f} yrs')
        return self

    # ── 4.3  Remove impossible physiological values ───────
    def remove_impossible_values(self):
        before = len(self.df)
        df = self.df.copy()

        # Age: 18-120 years
        if 'age' in df.columns:
            df = df[df['age'].between(18, 120)]

        # Blood pressure (Heart Disease dataset)
        if 'trestbps' in df.columns:
            df = df[df['trestbps'].between(60, 250)]

        # Blood pressure systolic / diastolic (Cardiovascular dataset)
        if 'ap_hi' in df.columns:
            df = df[df['ap_hi'].between(60, 300)]
        if 'ap_lo' in df.columns:
            df = df[df['ap_lo'].between(40, 200)]
        # Systolic must be > diastolic
        if 'ap_hi' in df.columns and 'ap_lo' in df.columns:
            df = df[df['ap_hi'] > df['ap_lo']]

        # Cholesterol (Heart Disease dataset, mg/dL)
        if 'chol' in df.columns:
            df = df[(df['chol'] > 100) & (df['chol'] < 600)]

        # Resting heart rate
        if 'thalach' in df.columns:
            df = df[df['thalach'].between(40, 220)]

        # Weight (kg) & Height (cm)
        if 'weight' in df.columns:
            df = df[df['weight'].between(30, 250)]
        if 'height' in df.columns:
            df = df[df['height'].between(100, 230)]

        # No negative values allowed anywhere
        num_cols = df.select_dtypes(include='number').columns
        for col in num_cols:
            if col not in ['target', 'cardio', 'exang']:
                df = df[df[col] >= 0]

        self.df = df
        self._record('impossible_values', before, len(self.df))
        return self

    # ── 4.4  Remove statistical outliers (IQR method) ────
    def remove_outliers_iqr(self, cols: Optional[list] = None,
                             factor: float = 3.5) -> 'MedicalDataCleaner':
        """
        Conservative IQR factor (3.5) — remove only extreme outliers
        to preserve rare but real medical cases.
        """
        before = len(self.df)
        if cols is None:
            cols = self.df.select_dtypes(include='number').columns.tolist()
            # Exclude binary / target cols
            exclude = ['target', 'cardio', 'sex', 'gender', 'fbs',
                       'exang', 'smoke', 'alco', 'active',
                       'cholesterol', 'gluc']
            cols = [c for c in cols if c not in exclude]

        mask = pd.Series([True] * len(self.df), index=self.df.index)
        for col in cols:
            Q1, Q3 = self.df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            lo  = Q1 - factor * IQR
            hi  = Q3 + factor * IQR
            mask &= self.df[col].between(lo, hi)

        self.df = self.df[mask]
        self._record(f'outliers_IQR_x{factor}', before, len(self.df))
        return self

    # ── 4.5  Handle remaining nulls ───────────────────────
    def handle_nulls(self, strategy: str = 'median') -> 'MedicalDataCleaner':
        """
        Drop rows where >30% values are null;
        impute remaining nulls with median (numeric) or mode (categorical).
        NOTE: This is TRAINING-time imputation. Inference uses a separate strategy.
        """
        thresh = int(0.7 * len(self.df.columns))
        before = len(self.df)
        self.df = self.df.dropna(thresh=thresh)
        self._record('drop_high_null_rows', before, len(self.df))

        # Impute remaining
        num_cols = self.df.select_dtypes(include='number').columns
        for col in num_cols:
            if self.df[col].isnull().any():
                self.df[col].fillna(self.df[col].median(), inplace=True)

        cat_cols = self.df.select_dtypes(exclude='number').columns
        for col in cat_cols:
            if self.df[col].isnull().any():
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)

        print(f'   [handle_nulls] Remaining nulls: {self.df.isnull().sum().sum()}')
        return self

    # ── 4.6  Encode categorical columns ──────────────────
    def encode_categoricals(self) -> 'MedicalDataCleaner':
        cat_cols = self.df.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            le = LabelEncoder()
            self.df[col] = le.fit_transform(self.df[col].astype(str))
            print(f'   [encode] {col}: {list(le.classes_)}')
        return self

    # ── 4.7  Reset index and finalize ─────────────────────
    def finalize(self) -> pd.DataFrame:
        self.df = self.df.reset_index(drop=True)
        print(f'\n   ✅ Cleaning complete: {self._original_shape} → {self.df.shape}')
        print(f'   Retained: {len(self.df)/self._original_shape[0]*100:.1f}%')
        return self.df

    def print_log(self):
        print(f'\n{"─"*55}')
        print(f'  Cleaning log — {self.name}')
        print(f'{"─"*55}')
        for entry in self.log:
            print(f"  {entry['step']:<30} removed {entry['removed']:>5}")


# ── Run cleaning ──────────────────────────────────────────
print('🧹 Cleaning Heart Disease dataset...')
cleaner_hd = (
    MedicalDataCleaner(df_hd, 'Heart Disease')
    .remove_duplicates()
    .remove_impossible_values()
    .remove_outliers_iqr()
    .handle_nulls()
    .encode_categoricals()
)
df_hd_clean = cleaner_hd.finalize()
cleaner_hd.print_log()

print('\n🧹 Cleaning Cardiovascular dataset...')
cleaner_cv = (
    MedicalDataCleaner(df_cv, 'Cardiovascular')
    .remove_duplicates()
    .fix_age_days_to_years()
    .remove_impossible_values()
    .remove_outliers_iqr()
    .handle_nulls()
    .encode_categoricals()
)
df_cv_clean = cleaner_cv.finalize()
cleaner_cv.print_log()

# ── Save cleaned datasets to Drive ────────────────────────
df_hd_clean.to_csv(f"{PATHS['cleaned']}/heart_disease_clean.csv", index=False)
df_cv_clean.to_csv(f"{PATHS['cleaned']}/cardiovascular_clean.csv", index=False)
print(f'\n💾 Cleaned datasets saved to: {PATHS["cleaned"]}')

In [ ]:
# ============================================================
# CELL 5 — Medical Feature Engineering
# ============================================================
# Transforms raw measurements into clinically meaningful
# representations used in cardiology practice.
# ============================================================

class MedicalFeatureEngineer:
    """
    Generate advanced medical features from raw measurements.
    All formulas are standard clinical definitions.
    """

    def __init__(self, df: pd.DataFrame, dataset_name: str):
        self.df   = df.copy()
        self.name = dataset_name
        self.new_features: List[str] = []

    def _add(self, col_name: str, values, description: str) -> None:
        self.df[col_name] = values
        self.new_features.append(col_name)
        print(f'   ✓ {col_name:<28} — {description}')

    # ─────────────────────────────────────────────────────
    # Features for CARDIOVASCULAR dataset
    # (has ap_hi, ap_lo, weight, height, cholesterol, gluc)
    # ─────────────────────────────────────────────────────

    def add_bmi(self):
        """BMI = weight(kg) / height(m)²"""
        if 'weight' in self.df.columns and 'height' in self.df.columns:
            h_m = self.df['height'] / 100.0
            bmi = self.df['weight'] / (h_m ** 2)
            self._add('bmi', bmi.round(2),
                      'Body Mass Index  [kg/m²]')

            # Obesity category (WHO)
            bins   = [0, 18.5, 25, 30, 35, 40, np.inf]
            labels = [0, 1, 2, 3, 4, 5]  # Underweight→Obese III
            self._add('bmi_category',
                      pd.cut(bmi, bins=bins, labels=labels).astype(float),
                      'WHO BMI category  (0=Underweight, 5=Obese III)')
        return self

    def add_blood_pressure_features(self):
        """MAP and Pulse Pressure from systolic/diastolic BP."""
        if 'ap_hi' in self.df.columns and 'ap_lo' in self.df.columns:
            s, d = self.df['ap_hi'], self.df['ap_lo']

            # Mean Arterial Pressure: MAP = DBP + (SBP-DBP)/3
            map_val = d + (s - d) / 3
            self._add('map', map_val.round(2),
                      'Mean Arterial Pressure  [mmHg]')

            # Pulse Pressure: PP = SBP - DBP
            self._add('pulse_pressure', (s - d).round(2),
                      'Pulse Pressure  [mmHg]  (arterial stiffness marker)')

            # Hypertension grade (ESC/ESH guidelines)
            # 0=Normal, 1=High-normal, 2=Grade1, 3=Grade2, 4=Grade3
            def ht_grade(row):
                sbp, dbp = row['ap_hi'], row['ap_lo']
                if sbp >= 180 or dbp >= 110: return 4
                if sbp >= 160 or dbp >= 100: return 3
                if sbp >= 140 or dbp >= 90:  return 2
                if sbp >= 130 or dbp >= 85:  return 1
                return 0
            self._add('hypertension_grade',
                      self.df.apply(ht_grade, axis=1),
                      'ESC/ESH Hypertension grade  (0-4)')

        # Heart Disease dataset uses trestbps
        if 'trestbps' in self.df.columns:
            self._add('bp_category',
                      pd.cut(self.df['trestbps'],
                             bins=[0, 120, 130, 140, 160, np.inf],
                             labels=[0, 1, 2, 3, 4]).astype(float),
                      'Resting BP category  (0=Normal … 4=Crisis)')
        return self

    def add_metabolic_risk_score(self):
        """
        Metabolic Risk Score = weighted combination of metabolic risk factors.
        Designed for the Cardiovascular dataset.
        """
        score = pd.Series(0.0, index=self.df.index)

        if 'bmi' in self.df.columns:
            score += (self.df['bmi'] > 30).astype(float) * 2.0   # Obesity
            score += (self.df['bmi'] > 35).astype(float) * 1.5   # Severe obesity

        if 'hypertension_grade' in self.df.columns:
            score += self.df['hypertension_grade'] * 1.5

        if 'cholesterol' in self.df.columns:
            # Cholesterol in CV dataset: 1=normal, 2=above normal, 3=well above
            score += (self.df['cholesterol'] - 1) * 2.0

        if 'gluc' in self.df.columns:
            score += (self.df['gluc'] - 1) * 1.5

        if 'smoke' in self.df.columns:
            score += self.df['smoke'] * 3.0

        if 'alco' in self.df.columns:
            score += self.df['alco'] * 1.5

        if score.sum() > 0:
            self._add('metabolic_risk_score', score.round(2),
                      'Composite metabolic risk  (higher = worse)')
        return self

    def add_cardiac_stress_index(self):
        """Cardiac Stress Index (Heart Disease dataset)."""
        if 'thalach' in self.df.columns and 'trestbps' in self.df.columns:
            # Double Product (Rate-Pressure Product): HR × SBP / 100
            # Reflects myocardial oxygen demand
            rpp = (self.df['thalach'] * self.df['trestbps']) / 100
            self._add('rate_pressure_product', rpp.round(2),
                      'Rate-Pressure Product  (myocardial O₂ demand)')

        if 'thalach' in self.df.columns and 'age' in self.df.columns:
            # Chronotropic Index: achieved HR / max predicted HR
            # Max predicted HR = 220 - age
            max_hr = 220 - self.df['age']
            ci = self.df['thalach'] / max_hr
            self._add('chronotropic_index', ci.round(3),
                      'Chronotropic Index  (exercise HR response)')

        if 'oldpeak' in self.df.columns and 'thalach' in self.df.columns:
            # ST-depression normalized by max HR
            self._add('st_depression_index',
                      (self.df['oldpeak'] / (self.df['thalach'] / 100)).round(3),
                      'ST-Depression Index  (ischemia marker)')
        return self

    def add_lifestyle_risk_score(self):
        """Lifestyle Risk Score for Cardiovascular dataset."""
        score = pd.Series(0.0, index=self.df.index)

        if 'smoke' in self.df.columns:
            score += self.df['smoke'] * 3.0
        if 'alco' in self.df.columns:
            score += self.df['alco'] * 2.0
        if 'active' in self.df.columns:
            score -= self.df['active'] * 1.5  # Active lifestyle = protective

        if score.abs().sum() > 0:
            self._add('lifestyle_risk_score', score.round(2),
                      'Lifestyle Risk Score  (positive = higher risk)')
        return self

    def add_age_risk_interaction(self):
        """Age × risk factor interaction terms."""
        if 'age' not in self.df.columns:
            return self

        if 'bmi' in self.df.columns:
            self._add('age_bmi_interaction',
                      (self.df['age'] * self.df['bmi'] / 100).round(3),
                      'Age × BMI interaction')

        if 'hypertension_grade' in self.df.columns:
            self._add('age_hypertension_interaction',
                      (self.df['age'] * self.df['hypertension_grade']).round(2),
                      'Age × Hypertension interaction')

        if 'oldpeak' in self.df.columns:
            self._add('age_st_depression',
                      (self.df['age'] * self.df['oldpeak']).round(3),
                      'Age × ST-depression interaction')
        return self

    def finalize(self) -> pd.DataFrame:
        print(f'\n   ✅ Feature engineering complete.')
        print(f'   New features ({len(self.new_features)}): {self.new_features}')
        print(f'   Final shape: {self.df.shape}')
        return self.df


# ── Engineer features ─────────────────────────────────────
print('⚗️  Engineering features for Heart Disease dataset...')
eng_hd = (
    MedicalFeatureEngineer(df_hd_clean, 'Heart Disease')
    .add_blood_pressure_features()
    .add_cardiac_stress_index()
    .add_age_risk_interaction()
)
df_hd_eng = eng_hd.finalize()

print('\n⚗️  Engineering features for Cardiovascular dataset...')
eng_cv = (
    MedicalFeatureEngineer(df_cv_clean, 'Cardiovascular')
    .add_bmi()
    .add_blood_pressure_features()
    .add_metabolic_risk_score()
    .add_lifestyle_risk_score()
    .add_age_risk_interaction()
)
df_cv_eng = eng_cv.finalize()

# ── Save engineered datasets ──────────────────────────────
df_hd_eng.to_csv(f"{PATHS['engineered']}/heart_disease_engineered.csv", index=False)
df_cv_eng.to_csv(f"{PATHS['engineered']}/cardiovascular_engineered.csv", index=False)
print(f'\n💾 Engineered datasets saved to: {PATHS["engineered"]}')

# ── Feature correlation with target ───────────────────────
def plot_feature_importance_corr(df, target_col, title, save_path):
    num_df = df.select_dtypes(include='number')
    corr = num_df.corr()[target_col].drop(target_col).sort_values()
    colors = ['#F44336' if v < 0 else '#2196F3' for v in corr.values]

    fig, ax = plt.subplots(figsize=(10, max(6, len(corr) * 0.35)))
    ax.barh(corr.index, corr.values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Feature–Target Correlation: {title}', fontweight='bold')
    ax.set_xlabel('Pearson Correlation with Target')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_feature_importance_corr(df_hd_eng, 'target', 'Heart Disease',
    f"{PATHS['plots']}/feature_corr_hd.png")
plot_feature_importance_corr(df_cv_eng, 'cardio', 'Cardiovascular',
    f"{PATHS['plots']}/feature_corr_cv.png")

In [ ]:
# ============================================================
# CELL 6 — Model Training
# ============================================================
# Trains XGBoost, CatBoost, LightGBM, Logistic Regression
# on BOTH datasets with 5-fold stratified cross-validation.
# ============================================================

def prepare_data(df: pd.DataFrame, target_col: str):
    """
    Split into X, y and perform stratified train/test split.
    Returns: X_train, X_test, y_train, y_test, feature_names
    """
    # Drop non-feature columns
    drop_cols = [target_col, 'id'] if 'id' in df.columns else [target_col]
    X = df.drop(columns=drop_cols)
    y = df[target_col].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    return X_train, X_test, y_train, y_test, list(X.columns)


def evaluate_model(model, X_test, y_test, name: str) -> Dict:
    """Compute comprehensive evaluation metrics."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] \
              if hasattr(model, 'predict_proba') else y_pred

    metrics = {
        'model'    : name,
        'roc_auc'  : roc_auc_score(y_test, y_proba),
        'f1'       : f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
    }
    return metrics


def train_all_models(X_train, X_test, y_train, y_test,
                     dataset_name: str, scaler=None) -> Dict:
    """
    Train all base models and evaluate via cross-validation.
    Returns dict of {model_name: trained_model} and metrics_df.
    """
    print(f'\n{"═"*60}')
    print(f'  Training on: {dataset_name}')
    print(f'  Train: {X_train.shape}  Test: {X_test.shape}')
    print(f'{"═"*60}')

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

    # ── Scale features for LR ─────────────────────────────
    if scaler is None:
        scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # ── Model definitions ─────────────────────────────────
    models = {
        'XGBoost': XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE, eval_metric='logloss',
            verbosity=0
        ),
        'CatBoost': CatBoostClassifier(
            iterations=400, depth=6, learning_rate=0.05,
            l2_leaf_reg=3, random_state=RANDOM_STATE,
            verbose=0, eval_metric='AUC'
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE, verbose=-1
        ),
        'LogisticRegression': LogisticRegression(
            max_iter=1000, C=1.0,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    trained_models = {}
    all_metrics    = []
    lr_models      = {'LogisticRegression'}  # use scaled data

    for name, model in models.items():
        X_tr = X_train_sc if name in lr_models else X_train.values
        X_te = X_test_sc  if name in lr_models else X_test.values

        print(f'\n  ▶ Training {name}...')
        # 5-fold CV AUC
        cv_scores = cross_val_score(
            model, X_tr, y_train, cv=cv,
            scoring='roc_auc', n_jobs=-1
        )
        print(f'    CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

        # Final fit on full training set
        model.fit(X_tr, y_train)
        trained_models[name] = model

        # Test set evaluation
        metrics = evaluate_model(model, X_te, y_test, name)
        metrics['cv_auc_mean'] = cv_scores.mean()
        metrics['cv_auc_std']  = cv_scores.std()
        all_metrics.append(metrics)
        print(f'    Test AUC:{metrics["roc_auc"]:.4f}  '
              f'F1:{metrics["f1"]:.4f}  '
              f'Prec:{metrics["precision"]:.4f}  '
              f'Rec:{metrics["recall"]:.4f}')

    metrics_df = pd.DataFrame(all_metrics).set_index('model')
    return trained_models, metrics_df, scaler, X_test, y_test, X_test_sc


# ── Prepare datasets ──────────────────────────────────────
Xtr_hd, Xte_hd, ytr_hd, yte_hd, feat_hd = prepare_data(df_hd_eng, 'target')
Xtr_cv, Xte_cv, ytr_cv, yte_cv, feat_cv = prepare_data(df_cv_eng, 'cardio')

# ── Train ─────────────────────────────────────────────────
models_hd, metrics_hd, scaler_hd, Xte_hd, yte_hd, Xte_hd_sc = \
    train_all_models(Xtr_hd, Xte_hd, ytr_hd, yte_hd, 'Heart Disease')

models_cv, metrics_cv, scaler_cv, Xte_cv, yte_cv, Xte_cv_sc = \
    train_all_models(Xtr_cv, Xte_cv, ytr_cv, yte_cv, 'Cardiovascular')

# ── Print comparison tables ───────────────────────────────
print('\n📊 Heart Disease — Model Comparison')
display(metrics_hd.round(4))
print('\n📊 Cardiovascular — Model Comparison')
display(metrics_cv.round(4))

# ── Plot ROC curves ───────────────────────────────────────
def plot_roc_curves(models_dict, X_test, y_test, lr_X_test_sc, title, save_path):
    lr_set = {'LogisticRegression'}
    fig, ax = plt.subplots(figsize=(9, 7))
    colors = ['#E91E63', '#2196F3', '#4CAF50', '#FF9800']

    for (name, model), color in zip(models_dict.items(), colors):
        X = lr_X_test_sc if name in lr_set else X_test.values
        proba = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {title}', fontweight='bold')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_roc_curves(models_hd, Xte_hd, yte_hd, Xte_hd_sc,
    'Heart Disease', f"{PATHS['plots']}/roc_heart_disease.png")
plot_roc_curves(models_cv, Xte_cv, yte_cv, Xte_cv_sc,
    'Cardiovascular', f"{PATHS['plots']}/roc_cardiovascular.png")

In [ ]:
# ============================================================
# CELL 7 — Ensemble: Soft Voting + Stacking
# ============================================================

def build_ensemble(trained_models: Dict, X_train, y_train,
                   X_test, y_test, dataset_name: str,
                   scaler, X_test_sc) -> Dict:
    """
    Build two ensemble strategies:
      1. Soft Voting   — average predicted probabilities
      2. Stacking      — meta-learner (LR) on top of base models
    """
    print(f'\n{"═"*60}')
    print(f'  Building Ensemble — {dataset_name}')
    print(f'{"═"*60}')

    lr_set = {'LogisticRegression'}

    # ── Manual Soft Voting ────────────────────────────────
    # Average probabilities from all base models
    probas = []
    for name, model in trained_models.items():
        X = X_test_sc if name in lr_set else X_test.values
        p = model.predict_proba(X)[:, 1]
        probas.append(p)

    avg_proba = np.mean(probas, axis=0)
    avg_pred  = (avg_proba >= 0.5).astype(int)

    voting_metrics = {
        'model'    : 'SoftVoting',
        'roc_auc'  : roc_auc_score(y_test, avg_proba),
        'f1'       : f1_score(y_test, avg_pred),
        'precision': precision_score(y_test, avg_pred),
        'recall'   : recall_score(y_test, avg_pred),
    }
    print(f'\n  ▶ Soft Voting')
    print(f'    AUC:{voting_metrics["roc_auc"]:.4f}  '
          f'F1:{voting_metrics["f1"]:.4f}  '
          f'Prec:{voting_metrics["precision"]:.4f}  '
          f'Rec:{voting_metrics["recall"]:.4f}')

    # ── Stacking ──────────────────────────────────────────
    # Use XGBoost, CatBoost, LightGBM as base; LR as meta
    # NOTE: For simplicity, train stacking on training set probabilities
    base_names = ['XGBoost', 'CatBoost', 'LightGBM']
    train_meta_features = np.column_stack([
        trained_models[n].predict_proba(X_train.values)[:, 1]
        for n in base_names
    ])
    test_meta_features = np.column_stack([
        trained_models[n].predict_proba(X_test.values)[:, 1]
        for n in base_names
    ])

    meta_learner = LogisticRegression(max_iter=500, C=1.0, random_state=RANDOM_STATE)
    meta_learner.fit(train_meta_features, y_train)

    stack_proba = meta_learner.predict_proba(test_meta_features)[:, 1]
    stack_pred  = meta_learner.predict(test_meta_features)

    stacking_metrics = {
        'model'    : 'Stacking',
        'roc_auc'  : roc_auc_score(y_test, stack_proba),
        'f1'       : f1_score(y_test, stack_pred),
        'precision': precision_score(y_test, stack_pred),
        'recall'   : recall_score(y_test, stack_pred),
    }
    print(f'\n  ▶ Stacking (XGB+Cat+LGB → LR meta)')
    print(f'    AUC:{stacking_metrics["roc_auc"]:.4f}  '
          f'F1:{stacking_metrics["f1"]:.4f}  '
          f'Prec:{stacking_metrics["precision"]:.4f}  '
          f'Rec:{stacking_metrics["recall"]:.4f}')

    # ── Confusion matrix ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pred, label in zip(axes,
                               [avg_pred, stack_pred],
                               ['Soft Voting', 'Stacking']):
        cm = confusion_matrix(y_test, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['No Disease', 'Disease'],
                    yticklabels=['No Disease', 'Disease'])
        ax.set_title(f'{label} — {dataset_name}', fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')

    plt.tight_layout()
    save_path = f"{PATHS['plots']}/confusion_{dataset_name.lower().replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

    return {
        'soft_voting_proba': avg_proba,
        'stacking_model'   : meta_learner,
        'voting_metrics'   : voting_metrics,
        'stacking_metrics' : stacking_metrics,
    }


ensemble_hd = build_ensemble(
    models_hd, Xtr_hd, ytr_hd, Xte_hd, yte_hd,
    'Heart Disease', scaler_hd, Xte_hd_sc
)
ensemble_cv = build_ensemble(
    models_cv, Xtr_cv, ytr_cv, Xte_cv, yte_cv,
    'Cardiovascular', scaler_cv, Xte_cv_sc
)

# ── Save models ───────────────────────────────────────────
for name, model in models_hd.items():
    joblib.dump(model, f"{PATHS['models']}/hd_{name.lower()}.pkl")
for name, model in models_cv.items():
    joblib.dump(model, f"{PATHS['models']}/cv_{name.lower()}.pkl")

joblib.dump(scaler_hd, f"{PATHS['models']}/hd_scaler.pkl")
joblib.dump(scaler_cv, f"{PATHS['models']}/cv_scaler.pkl")
joblib.dump(ensemble_hd['stacking_model'], f"{PATHS['models']}/hd_stack_meta.pkl")
joblib.dump(ensemble_cv['stacking_model'], f"{PATHS['models']}/cv_stack_meta.pkl")

# Save feature name lists
with open(f"{PATHS['models']}/hd_features.json", 'w') as f:
    json.dump(feat_hd, f)
with open(f"{PATHS['models']}/cv_features.json", 'w') as f:
    json.dump(feat_cv, f)

print(f'\n💾 All models saved to: {PATHS["models"]}')

In [ ]:
# ============================================================
# CELL 8 — SHAP Explainability
# ============================================================
# SHAP (SHapley Additive exPlanations) quantifies each feature's
# contribution to individual predictions and globally.
# ============================================================

shap.initjs()

def compute_and_plot_shap(
        model, X_test: pd.DataFrame, model_name: str,
        dataset_name: str, max_display: int = 15) -> np.ndarray:
    """
    Compute SHAP values and generate:
      1. Summary plot (beeswarm)
      2. Bar plot (mean |SHAP|)
    Returns SHAP values array.
    """
    print(f'\n  Computing SHAP for {model_name} — {dataset_name}...')

    # Use TreeExplainer for tree-based models
    if model_name in ['XGBoost', 'CatBoost', 'LightGBM']:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        # Some models return list [neg, pos]; take positive class
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    else:
        explainer = shap.LinearExplainer(model, X_test)
        shap_values = explainer.shap_values(X_test)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    # ── Summary plot (beeswarm) ────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, X_test, max_display=max_display,
                      show=False, plot_type='dot')
    plt.title(f'SHAP Summary — {model_name} ({dataset_name})',
              fontweight='bold', pad=15)
    plt.tight_layout()
    save_path = (f"{PATHS['plots']}/shap_summary_"
                 f"{dataset_name.lower().replace(' ', '_')}_"
                 f"{model_name.lower()}.png")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 {save_path}')

    # ── Bar plot (mean absolute SHAP) ─────────────────────
    mean_shap = np.abs(shap_values).mean(axis=0)
    feat_shap = pd.Series(mean_shap, index=X_test.columns)\
                  .sort_values(ascending=True).tail(max_display)

    fig, ax = plt.subplots(figsize=(9, max(5, len(feat_shap) * 0.4)))
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feat_shap)))
    ax.barh(feat_shap.index, feat_shap.values, color=colors)
    ax.set_title(f'Feature Importance (|SHAP|) — {model_name} ({dataset_name})',
                 fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    save_path2 = save_path.replace('summary', 'importance')
    plt.savefig(save_path2, dpi=120, bbox_inches='tight')
    plt.show()

    return shap_values


def explain_single_patient(
        model, explainer_data: pd.DataFrame, patient_row: pd.Series,
        model_name: str, dataset_name: str) -> Dict:
    """
    Generate a local SHAP waterfall explanation for ONE patient.
    Returns top risk factors with their SHAP contributions.
    """
    if model_name in ['XGBoost', 'CatBoost', 'LightGBM']:
        explainer = shap.TreeExplainer(model)
        patient_df = patient_row.to_frame().T
        sv = explainer.shap_values(patient_df)
        if isinstance(sv, list): sv = sv[1]
        sv = sv[0]
        base_val = explainer.expected_value
        if isinstance(base_val, (list, np.ndarray)):
            base_val = base_val[1] if len(base_val) > 1 else base_val[0]
    else:
        explainer = shap.LinearExplainer(model, explainer_data)
        patient_df = patient_row.to_frame().T
        sv = explainer.shap_values(patient_df)
        if isinstance(sv, list): sv = sv[1]
        sv = sv[0]
        base_val = explainer.expected_value
        if isinstance(base_val, (list, np.ndarray)):
            base_val = base_val[1] if len(base_val) > 1 else base_val[0]

    feature_contributions = pd.Series(sv, index=patient_row.index)\
                             .sort_values(key=abs, ascending=False)

    print(f'\n  🔍 Local Explanation ({model_name} — {dataset_name})')
    print(f'  Base value (avg prediction): {float(base_val):.3f}')
    print(f'  Top risk-increasing features:')
    for feat, val in feature_contributions.head(5).items():
        direction = '⬆️ increases risk' if val > 0 else '⬇️ decreases risk'
        print(f'    {feat:<30} SHAP={val:+.4f}  {direction}')

    return {
        'shap_values'    : sv,
        'base_value'     : float(base_val),
        'contributions'  : feature_contributions.to_dict(),
    }


# ── Compute global SHAP for best model (XGBoost) on both datasets ─
print('🧠 Computing SHAP explanations...')

# Use a subsample for speed (500 rows)
N_SHAP = min(500, len(Xte_hd))
Xte_hd_sample = Xte_hd.iloc[:N_SHAP]
shap_vals_hd = compute_and_plot_shap(
    models_hd['XGBoost'], Xte_hd_sample, 'XGBoost', 'Heart Disease')

N_SHAP_CV = min(500, len(Xte_cv))
Xte_cv_sample = Xte_cv.iloc[:N_SHAP_CV]
shap_vals_cv = compute_and_plot_shap(
    models_cv['XGBoost'], Xte_cv_sample, 'XGBoost', 'Cardiovascular')

# ── Local explanation: pick first high-risk test patient ──
print('\n── Local patient explanation (first high-risk patient) ──')
high_risk_idx_hd = yte_hd[yte_hd == 1].index[0]
patient_explanation_hd = explain_single_patient(
    models_hd['XGBoost'], Xte_hd,
    Xte_hd.loc[high_risk_idx_hd], 'XGBoost', 'Heart Disease'
)

In [ ]:
# ============================================================
# CELL 9 — Robust Real-World Inference Pipeline
# ============================================================
#
# KEY DESIGN PRINCIPLES:
#   1. Never crash on missing fields
#   2. Intelligent inference-time imputation
#   3. Confidence score that reflects data completeness
#   4. Clean interface for future Fuzzy Logic fusion
#
# FUZZY FUSION HOOK:
#   predict_patient() returns a dict with an 'ml_result' key.
#   The Fuzzy engine will receive this dict, add its own
#   'fuzzy_result' key, and compute a 'hybrid_score'.
# ============================================================

class CardiovascularInferencePipeline:
    """
    Production-grade inference engine.
    Handles missing values gracefully at runtime.
    """

    # ── Clinical reference medians (population-level fallbacks) ──
    # Used ONLY when patient data is missing AND no training median is available.
    CLINICAL_MEDIANS = {
        # Heart Disease dataset features
        'age'      : 55.0,  'sex'    : 1.0,  'cp'     : 0.0,
        'trestbps' : 130.0, 'chol'   : 240.0,'fbs'    : 0.0,
        'restecg'  : 0.0,   'thalach': 150.0,'exang'  : 0.0,
        'oldpeak'  : 0.8,   'slope'  : 1.0,  'ca'     : 0.0,
        'thal'     : 2.0,
        # Cardiovascular dataset features
        'gender'   : 1.0,   'height' : 165.0,'weight' : 74.0,
        'ap_hi'    : 120.0, 'ap_lo'  : 80.0,
        'cholesterol': 1.0, 'gluc'   : 1.0,
        'smoke'    : 0.0,   'alco'   : 0.0,  'active' : 1.0,
    }

    def __init__(self, models_hd: Dict, models_cv: Dict,
                 scaler_hd, scaler_cv,
                 feat_hd: List[str], feat_cv: List[str],
                 train_medians_hd: Dict = None,
                 train_medians_cv: Dict = None):
        self.models_hd = models_hd
        self.models_cv = models_cv
        self.scaler_hd = scaler_hd
        self.scaler_cv = scaler_cv
        self.feat_hd   = feat_hd
        self.feat_cv   = feat_cv
        # Training set medians (best imputation source)
        self.train_med_hd = train_medians_hd or {}
        self.train_med_cv = train_medians_cv or {}

    # ── Feature engineering (mirrors Cell 5) ─────────────
    @staticmethod
    def _engineer_features_hd(raw: Dict) -> Dict:
        """Apply HD feature engineering to raw patient dict."""
        d = dict(raw)
        if 'trestbps' in d:
            d['bp_category'] = min(4, max(0,
                int(pd.cut([d['trestbps']],
                   bins=[0,120,130,140,160,np.inf],
                   labels=[0,1,2,3,4])[0])))
        if 'thalach' in d and 'trestbps' in d:
            d['rate_pressure_product'] = d['thalach'] * d['trestbps'] / 100
        if 'thalach' in d and 'age' in d:
            d['chronotropic_index'] = d['thalach'] / (220 - d['age'])
        if 'oldpeak' in d and 'thalach' in d and d['thalach'] > 0:
            d['st_depression_index'] = d['oldpeak'] / (d['thalach'] / 100)
        if 'age' in d and 'oldpeak' in d:
            d['age_st_depression'] = d['age'] * d['oldpeak']
        return d

    @staticmethod
    def _engineer_features_cv(raw: Dict) -> Dict:
        """Apply CV feature engineering to raw patient dict."""
        d = dict(raw)
        if 'weight' in d and 'height' in d and d['height'] > 0:
            h_m = d['height'] / 100
            d['bmi'] = d['weight'] / (h_m ** 2)
            # BMI category
            for thresh, cat in [(18.5,0),(25,1),(30,2),(35,3),(40,4)]:
                if d['bmi'] < thresh:
                    d['bmi_category'] = float(cat)
                    break
            else:
                d['bmi_category'] = 5.0

        if 'ap_hi' in d and 'ap_lo' in d:
            s, dv = d['ap_hi'], d['ap_lo']
            d['map']           = dv + (s - dv) / 3
            d['pulse_pressure']= s - dv
            if   s >= 180 or dv >= 110: d['hypertension_grade'] = 4
            elif s >= 160 or dv >= 100: d['hypertension_grade'] = 3
            elif s >= 140 or dv >= 90:  d['hypertension_grade'] = 2
            elif s >= 130 or dv >= 85:  d['hypertension_grade'] = 1
            else:                        d['hypertension_grade'] = 0

        # Metabolic risk score
        score = 0.0
        if 'bmi' in d:                 score += (d['bmi'] > 30) * 2 + (d['bmi'] > 35) * 1.5
        if 'hypertension_grade' in d:  score += d['hypertension_grade'] * 1.5
        if 'cholesterol' in d:         score += (d['cholesterol'] - 1) * 2
        if 'gluc' in d:                score += (d['gluc'] - 1) * 1.5
        if 'smoke' in d:               score += d['smoke'] * 3
        if 'alco' in d:                score += d['alco'] * 1.5
        d['metabolic_risk_score'] = score

        # Lifestyle score
        ls = 0.0
        if 'smoke'  in d: ls += d['smoke'] * 3
        if 'alco'   in d: ls += d['alco'] * 2
        if 'active' in d: ls -= d['active'] * 1.5
        d['lifestyle_risk_score'] = ls

        if 'age' in d:
            if 'bmi' in d:
                d['age_bmi_interaction'] = d['age'] * d['bmi'] / 100
            if 'hypertension_grade' in d:
                d['age_hypertension_interaction'] = d['age'] * d['hypertension_grade']
        return d

    # ── Core imputation logic ─────────────────────────────
    def _impute_missing(self, engineered: Dict, feature_list: List[str],
                        train_medians: Dict) -> Tuple[np.ndarray, float, List[str]]:
        """
        Build the feature vector with graceful missing-value handling.

        Strategy (in priority order):
          1. Use provided value if present
          2. Impute with training-set median
          3. Impute with clinical population median
          4. Impute with 0.0 as last resort

        Returns:
          vector         — np.ndarray ready for model.predict_proba
          confidence     — fraction of features that were actually provided
          missing_list   — list of feature names that were imputed
        """
        vector       = []
        missing_list = []

        for feat in feature_list:
            if feat in engineered and engineered[feat] is not None \
               and not (isinstance(engineered[feat], float)
                        and np.isnan(engineered[feat])):
                vector.append(float(engineered[feat]))
            elif feat in train_medians:
                vector.append(float(train_medians[feat]))
                missing_list.append(feat)
            elif feat in self.CLINICAL_MEDIANS:
                vector.append(float(self.CLINICAL_MEDIANS[feat]))
                missing_list.append(feat)
            else:
                vector.append(0.0)
                missing_list.append(feat)

        n_provided  = len(feature_list) - len(missing_list)
        confidence  = n_provided / max(len(feature_list), 1)
        return np.array(vector).reshape(1, -1), confidence, missing_list

    # ── Predict on a single dataset ───────────────────────
    def _predict_dataset(self, patient_raw: Dict,
                         dataset: str) -> Dict:
        """
        Run ensemble prediction for one dataset.
        dataset: 'hd' or 'cv'
        """
        if dataset == 'hd':
            eng      = self._engineer_features_hd(patient_raw)
            feats    = self.feat_hd
            models   = self.models_hd
            scaler   = self.scaler_hd
            medians  = self.train_med_hd
        else:
            eng      = self._engineer_features_cv(patient_raw)
            feats    = self.feat_cv
            models   = self.models_cv
            scaler   = self.scaler_cv
            medians  = self.train_med_cv

        vec, conf, missing = self._impute_missing(eng, feats, medians)

        # Ensemble: average probabilities
        probas = []
        for name, model in models.items():
            X = scaler.transform(vec) if name == 'LogisticRegression' else vec
            p = model.predict_proba(X)[0, 1]
            probas.append(p)

        avg_proba = float(np.mean(probas))
        return {
            'probability'    : avg_proba,
            'confidence'     : conf,
            'missing_features': missing,
            'individual_probs': dict(zip(models.keys(), probas)),
        }

    # ── Public API ────────────────────────────────────────
    def predict_patient(self, patient_data: Dict) -> Dict:
        """
        Main inference entry point.

        Input:
            patient_data: dict with ANY subset of available features.
                          Missing features are handled gracefully.

        Output: {
            'ml_result': {
                'disease_probability' : float [0-1],
                'cardiovascular_risk_score': float [0-100],
                'severity_estimate'  : str,
                'confidence_score'   : float [0-1],
                'top_risk_factors'   : list[str],
                'missing_features'   : list[str],
                'individual_models'  : dict,
            },
            # FUZZY FUSION HOOK:
            # fuzzy_result will be added here by the Fuzzy engine
            # hybrid_score will be computed by the fusion layer
        }
        """
        results_by_ds = {}

        # Determine which dataset(s) to run
        has_hd_features = any(f in patient_data
                              for f in ['trestbps', 'thalach', 'cp', 'oldpeak', 'chol'])
        has_cv_features = any(f in patient_data
                              for f in ['ap_hi', 'ap_lo', 'height', 'weight', 'gluc'])

        # Always attempt both; engine degrades gracefully
        try:
            results_by_ds['hd'] = self._predict_dataset(patient_data, 'hd')
        except Exception as e:
            results_by_ds['hd'] = {'probability': 0.5, 'confidence': 0.0,
                                   'error': str(e)}
        try:
            results_by_ds['cv'] = self._predict_dataset(patient_data, 'cv')
        except Exception as e:
            results_by_ds['cv'] = {'probability': 0.5, 'confidence': 0.0,
                                   'error': str(e)}

        # ── Combine results ───────────────────────────────
        hd = results_by_ds['hd']
        cv = results_by_ds['cv']

        hd_conf = hd.get('confidence', 0)
        cv_conf = cv.get('confidence', 0)
        total_conf = hd_conf + cv_conf

        if total_conf > 0:
            # Confidence-weighted average
            combined_prob = (
                hd['probability'] * hd_conf +
                cv['probability'] * cv_conf
            ) / total_conf
        else:
            combined_prob = 0.5

        avg_confidence = (hd_conf + cv_conf) / 2

        # Risk score [0-100]
        risk_score = round(combined_prob * 100, 1)

        # Severity (calibrated thresholds)
        if combined_prob >= 0.70:   severity = 'HIGH'   
        elif combined_prob >= 0.45: severity = 'MODERATE'
        else:                       severity = 'LOW'

        # All missing features
        all_missing = list(set(
            hd.get('missing_features', []) +
            cv.get('missing_features', [])
        ))

        # Individual model breakdown
        individual_models = {}
        if 'individual_probs' in hd:
            individual_models.update(
                {f'HD_{k}': v for k, v in hd['individual_probs'].items()})
        if 'individual_probs' in cv:
            individual_models.update(
                {f'CV_{k}': v for k, v in cv['individual_probs'].items()})

        # Top contributing risk factors (from patient data)
        risk_factor_rules = {
            'age'             : lambda v: v > 65,
            'trestbps'        : lambda v: v > 140,
            'ap_hi'           : lambda v: v > 140,
            'chol'            : lambda v: v > 240,
            'fbs'             : lambda v: v == 1,
            'exang'           : lambda v: v == 1,
            'oldpeak'         : lambda v: v > 2,
            'smoke'           : lambda v: v == 1,
            'alco'            : lambda v: v == 1,
            'active'          : lambda v: v == 0,
            'bmi'             : lambda v: v > 30,
            'hypertension_grade': lambda v: v >= 2,
        }
        top_risk_factors = []
        eng_combined = {**self._engineer_features_hd(patient_data),
                        **self._engineer_features_cv(patient_data)}
        for feat, rule in risk_factor_rules.items():
            val = eng_combined.get(feat)
            if val is not None:
                try:
                    if rule(val):
                        top_risk_factors.append(feat)
                except Exception:
                    pass

        ml_result = {
            'disease_probability'      : round(combined_prob, 4),
            'cardiovascular_risk_score': risk_score,
            'severity_estimate'        : severity,
            'confidence_score'         : round(avg_confidence, 3),
            'top_risk_factors'         : top_risk_factors,
            'missing_features'         : all_missing,
            'individual_models'        : individual_models,
            'hd_probability'           : round(hd['probability'], 4),
            'cv_probability'           : round(cv['probability'], 4),
        }

        # ══════════════════════════════════════════════════
        # FUZZY FUSION HOOK
        # ══════════════════════════════════════════════════
        # To integrate the Fuzzy engine:
        #
        #   from fuzzy_engine import evaluate_patient
        #   fuzzy_result = evaluate_patient(echo_data)  # separate input
        #
        #   # Weighted fusion
        #   ml_prob    = ml_result['disease_probability']
        #   fuzzy_norm = fuzzy_result['score'] / 100.0
        #   ml_conf    = ml_result['confidence_score']
        #
        #   hybrid_score = (ml_prob * ml_conf + fuzzy_norm * (1 - ml_conf))
        # ══════════════════════════════════════════════════

        return {
            'ml_result'    : ml_result,
            'fuzzy_result' : None,   # placeholder
            'hybrid_score' : None,   # placeholder
        }


# ── Build training medians for imputation ─────────────────
train_medians_hd = Xtr_hd.median().to_dict()
train_medians_cv = Xtr_cv.median().to_dict()

# ── Instantiate pipeline ──────────────────────────────────
inference_pipeline = CardiovascularInferencePipeline(
    models_hd       = models_hd,
    models_cv       = models_cv,
    scaler_hd       = scaler_hd,
    scaler_cv       = scaler_cv,
    feat_hd         = feat_hd,
    feat_cv         = feat_cv,
    train_medians_hd= train_medians_hd,
    train_medians_cv= train_medians_cv,
)

# ── Test Case 1: Complete patient data ────────────────────
print('\n' + '═'*60)
print('  INFERENCE TEST — COMPLETE DATA')
print('═'*60)
patient_full = {
    # Heart Disease features
    'age': 62, 'sex': 1, 'cp': 3, 'trestbps': 158,
    'chol': 294, 'fbs': 1, 'restecg': 0, 'thalach': 106,
    'exang': 1, 'oldpeak': 2.8, 'slope': 1, 'ca': 2, 'thal': 3,
    # Cardiovascular features
    'gender': 1, 'height': 172, 'weight': 90,
    'ap_hi': 158, 'ap_lo': 98, 'cholesterol': 2,
    'gluc': 1, 'smoke': 0, 'alco': 0, 'active': 0
}
result_full = inference_pipeline.predict_patient(patient_full)
r = result_full['ml_result']
print(f"  Disease Probability  : {r['disease_probability']:.1%}")
print(f"  Risk Score           : {r['cardiovascular_risk_score']}/100")
print(f"  Severity             : {r['severity_estimate']}")
print(f"  Confidence           : {r['confidence_score']:.1%}")
print(f"  Top Risk Factors     : {r['top_risk_factors']}")
print(f"  Missing Features     : {r['missing_features'][:5]}...")

# ── Test Case 2: Partial patient (only BP + age) ──────────
print('\n' + '═'*60)
print('  INFERENCE TEST — PARTIAL DATA (BP + age only)')
print('═'*60)
patient_partial = {'age': 55, 'ap_hi': 148, 'ap_lo': 92, 'smoke': 1}
result_partial = inference_pipeline.predict_patient(patient_partial)
r2 = result_partial['ml_result']
print(f"  Disease Probability  : {r2['disease_probability']:.1%}")
print(f"  Risk Score           : {r2['cardiovascular_risk_score']}/100")
print(f"  Severity             : {r2['severity_estimate']}")
print(f"  Confidence           : {r2['confidence_score']:.1%}  ← low (most data missing)")
print(f"  Missing Features     : {len(r2['missing_features'])} features imputed")

# ── Test Case 3: Empty input (edge case) ──────────────────
print('\n' + '═'*60)
print('  INFERENCE TEST — EMPTY INPUT (edge case)')
print('═'*60)
result_empty = inference_pipeline.predict_patient({})
print(f"  Severity: {result_empty['ml_result']['severity_estimate']}")
print(f"  Confidence: {result_empty['ml_result']['confidence_score']:.1%}")
print('  ✅ Pipeline did not crash.')

In [ ]:
# ============================================================
# CELL 10 — Medical Report Generator
# ============================================================
# Generates a professional clinical-style text report
# and visualization dashboard for each patient.
# ============================================================

def generate_medical_report(patient_data: Dict,
                             inference_result: Dict,
                             patient_id: str = 'UNKNOWN',
                             save_dir: str = None) -> str:
    """
    Generate a professional medical AI report.

    Output:
      - Formatted text report (returned as string)
      - Dashboard visualization (saved to save_dir)
    """
    r         = inference_result['ml_result']
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')

    # ── Risk color coding ─────────────────────────────────
    severity_colors = {'LOW': '🟢', 'MODERATE': '🟡', 'HIGH': '🔴'}
    sev_icon = severity_colors.get(r['severity_estimate'], '⚪')

    # ── Build text report ─────────────────────────────────
    lines = [
        '=' * 65,
        '  HYBRID CARDIOVASCULAR AI ANALYSIS REPORT',
        '  Powered by Ensemble ML + Explainable AI',
        '=' * 65,
        f'  Patient ID   : {patient_id}',
        f'  Report Date  : {timestamp}',
        f'  Analysis Type: Machine Learning Ensemble (Fuzzy pending)',
        '-' * 65,
        '',
        '  ── PRIMARY FINDINGS ──────────────────────────────────',
        f'  Disease Probability    :  {r["disease_probability"]:.1%}',
        f'  Cardiovascular Risk    :  {r["cardiovascular_risk_score"]:.1f} / 100',
        f'  Severity Classification:  {sev_icon}  {r["severity_estimate"]}',
        f'  Confidence Score       :  {r["confidence_score"]:.1%}',
        '',
        '  ── DATASET-SPECIFIC PROBABILITIES ───────────────────',
        f'  Heart Disease (ECG/Stress model)  :  {r["hd_probability"]:.1%}',
        f'  Cardiovascular (BP/Lifestyle model):  {r["cv_probability"]:.1%}',
        '',
        '  ── INDIVIDUAL MODEL PREDICTIONS ─────────────────────',
    ]

    for model_name, prob in r['individual_models'].items():
        bar_len = int(prob * 20)
        bar = '█' * bar_len + '░' * (20 - bar_len)
        lines.append(f'  {model_name:<28} [{bar}] {prob:.1%}')

    lines += [
        '',
        '  ── CLINICAL RISK FACTORS DETECTED ──────────────────',
    ]
    if r['top_risk_factors']:
        for rf in r['top_risk_factors']:
            val = patient_data.get(rf, '—')
            lines.append(f'  ⚠️  {rf:<25} value: {val}')
    else:
        lines.append('  ✓  No major risk factors detected in provided data')

    lines += [
        '',
        '  ── DATA COMPLETENESS ────────────────────────────────',
        f'  Features provided      :  {len(patient_data)}',
        f'  Features imputed       :  {len(r["missing_features"])}',
        f'  Confidence penalty     :  {(1-r["confidence_score"]):.1%} reduction',
    ]

    if r['missing_features']:
        missing_preview = r['missing_features'][:6]
        lines.append(f'  Key missing features   :  {", ".join(missing_preview)}')

    lines += [
        '',
        '  ── CLINICAL RECOMMENDATION ──────────────────────────',
    ]

    if r['severity_estimate'] == 'HIGH':
        lines += [
            '  🔴 HIGH RISK — Immediate cardiology referral recommended.',
            '     Further testing: ECG, Echo, stress test, lipid panel.',
            '     Consider: antihypertensive, statin, antiplatelet therapy.',
        ]
    elif r['severity_estimate'] == 'MODERATE':
        lines += [
            '  🟡 MODERATE RISK — Cardiology follow-up within 4 weeks.',
            '     Lifestyle modification: diet, exercise, smoking cessation.',
            '     Monitor BP and cholesterol every 3 months.',
        ]
    else:
        lines += [
            '  🟢 LOW RISK — Routine annual cardiovascular screening.',
            '     Maintain healthy lifestyle and current medication.',
        ]

    lines += [
        '',
        '  ── FUZZY LOGIC INTEGRATION STATUS ──────────────────',
        '  Echocardiography analysis  :  PENDING (separate input)',
        '  Hybrid fusion score        :  PENDING',
        '  (Fuzzy engine will be integrated in Phase 2)',
        '',
        '  ── DISCLAIMER ──────────────────────────────────────',
        '  This report is generated by an AI system for research',
        '  and educational purposes. It does NOT replace clinical',
        '  judgment. All findings must be validated by a qualified',
        '  cardiologist before any medical decision is made.',
        '=' * 65,
    ]

    report_text = '\n'.join(lines)

    # ── Dashboard visualization ────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f'Cardiovascular AI Dashboard — Patient {patient_id}',
                 fontsize=16, fontweight='bold', y=0.98)

    gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

    # ── Panel 1: Risk gauge ────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    risk_val = r['cardiovascular_risk_score']
    color = '#F44336' if risk_val >= 70 else '#FF9800' if risk_val >= 45 else '#4CAF50'
    ax1.barh(['Risk Score'], [risk_val], color=color, height=0.4)
    ax1.barh(['Risk Score'], [100 - risk_val], left=risk_val,
             color='#E0E0E0', height=0.4)
    ax1.set_xlim(0, 100)
    ax1.set_title('Cardiovascular Risk Score', fontweight='bold')
    ax1.text(risk_val / 2, 0, f'{risk_val:.0f}', ha='center',
             va='center', fontsize=16, fontweight='bold', color='white')
    ax1.text(50, -0.35, r['severity_estimate'], ha='center',
             fontsize=12, fontweight='bold', color=color)
    ax1.set_yticks([])

    # ── Panel 2: Model consensus ───────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    model_names = list(r['individual_models'].keys())
    model_probs = list(r['individual_models'].values())
    colors_bar  = ['#F44336' if p >= 0.5 else '#4CAF50' for p in model_probs]
    ax2.barh(model_names, model_probs, color=colors_bar)
    ax2.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold 0.5')
    ax2.set_xlim(0, 1)
    ax2.set_title('Model Consensus', fontweight='bold')
    ax2.set_xlabel('Predicted Probability')
    ax2.legend(fontsize=8)

    # ── Panel 3: Data completeness pie ────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    n_provided = len(patient_data)
    n_imputed  = len(r['missing_features'])
    n_total    = n_provided + n_imputed
    if n_total > 0:
        wedge_sizes   = [n_provided, n_imputed]
        wedge_labels  = [f'Provided\n({n_provided})', f'Imputed\n({n_imputed})']
        wedge_colors  = ['#2196F3', '#FF9800']
        ax3.pie(wedge_sizes, labels=wedge_labels, colors=wedge_colors,
                autopct='%1.0f%%', startangle=90,
                textprops={'fontsize': 9})
    ax3.set_title('Data Completeness', fontweight='bold')

    # ── Panel 4: Risk factors bar ──────────────────────────
    ax4 = fig.add_subplot(gs[1, 0:2])
    risk_metrics = {
        'HD Probability': r['hd_probability'],
        'CV Probability': r['cv_probability'],
        'Combined Prob' : r['disease_probability'],
        'Confidence'    : r['confidence_score'],
    }
    bar_colors = ['#E91E63', '#2196F3', '#9C27B0', '#607D8B']
    bars = ax4.bar(risk_metrics.keys(), risk_metrics.values(),
                   color=bar_colors)
    ax4.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Risk threshold 0.5')
    ax4.set_ylim(0, 1.0)
    ax4.set_title('Key Metrics Summary', fontweight='bold')
    ax4.set_ylabel('Score (0–1)')
    ax4.legend(fontsize=8)
    for bar, val in zip(bars, risk_metrics.values()):
        ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # ── Panel 5: Report text summary ──────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis('off')
    summary_lines = [
        f"Patient: {patient_id}",
        f"Date: {timestamp}",
        '─' * 28,
        f"Risk: {r['cardiovascular_risk_score']:.0f}/100",
        f"Severity: {r['severity_estimate']}",
        f"Confidence: {r['confidence_score']:.0%}",
        '─' * 28,
        'Risk factors:',
    ]
    for rf in (r['top_risk_factors'][:4] or ['None detected']):
        summary_lines.append(f'  ⚠ {rf}')
    summary_lines.append('─' * 28)
    summary_lines.append('Fuzzy: PENDING')

    ax5.text(0.05, 0.95, '\n'.join(summary_lines),
             transform=ax5.transAxes,
             verticalalignment='top',
             fontfamily='monospace', fontsize=8.5,
             bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
    ax5.set_title('Summary Card', fontweight='bold')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig_path = f"{save_dir}/dashboard_{patient_id}.png"
        plt.savefig(fig_path, dpi=130, bbox_inches='tight')
        print(f'   💾 Dashboard: {fig_path}')

    plt.show()

    # Save text report
    if save_dir:
        report_path = f"{save_dir}/report_{patient_id}.txt"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        print(f'   💾 Report:    {report_path}')

    return report_text


# ── Generate reports ──────────────────────────────────────
print('📋 Generating medical reports...\n')

report1 = generate_medical_report(
    patient_full, result_full,
    patient_id='PT-2024-001',
    save_dir=PATHS['reports']
)
print(report1)

print('\n' + '─'*65 + '\n')

report2 = generate_medical_report(
    patient_partial, result_partial,
    patient_id='PT-2024-002-PARTIAL',
    save_dir=PATHS['reports']
)
print(report2)

print('\n✅ All reports saved to:', PATHS['reports'])
print('\n🎓 Pipeline complete. Ready for academic presentation.')
print('\nNext step: Integrate fuzzy_engine.evaluate_patient()')
print('           and populate fuzzy_result + hybrid_score in predict_patient().')